# Phase 3 of 3: Text CNN + BiGRU with Injection and Reason Heads

**Run order:** 3 of 4. Needs the Phase 1 outputs; compare its locked numbers against the Phase 2 winner.
**Produces:** `models/phase3-cnn-bigru-v1-<fingerprint>/` with weights, vocab, tokenizer config, thresholds, metrics, history and the run manifest. The directory must not exist yet; a rerun is a new run.
**Expect:** minutes per epoch on CUDA, early stopping around epoch 3 to 8, tens of minutes per epoch on CPU. Section 6 prints the CPU latency and whether the 100 ms p95 target for a batch of ten is met.

The `email-classifier-nn` Phase 3 architecture adapted per `docs/PLAN.md` §3 Stage B: a cased,
punctuation-preserving tokenizer over a train-only vocabulary, a 1D-CNN feeding a bidirectional
GRU with masked max-pooling, and **two heads**: a binary injection logit and a nine-way reason
logit. The reason loss is masked to positive rows; a reason is reported only when the binary head
fires.

Discipline: `train` fits; `val` alone drives early stopping (binary log loss), threshold selection
and reporting during development; `test` and `adversarial_test` are opened once in section 5. The
Phase 2 winner on the same splits is the bar. Every locked number carries a 95% bootstrap interval,
a recall-at-FPR table, and per-source and per-shape slices. Reason metrics are computed on true
positives only; reasons with no training rows are reported as undefined, never averaged in
(`docs/EVALUATION.md` §3.2).

Section 6 records parameter count, size on disk, and CPU latency for a batch of ten 512-token
chunks, the filter's deployment shape (`docs/EVALUATION.md` §8).

Runs unchanged on CUDA, Apple MPS, or CPU. Output: `models/phase3-cnn-bigru-v1-<fingerprint>/`.


## 1. Environment and Phase 1 Artifacts


In [1]:
import gc
import json
import os
import platform
import random
import re
import sys
import time
from collections import Counter
from contextlib import nullcontext
from copy import deepcopy
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    try:
        torch.cuda.set_per_process_memory_fraction(0.55, 0)
    except RuntimeError as exc:
        print(f"CUDA memory cap was not applied: {exc}")
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATASET_VERSION = "v4"  # v1 to v3 are superseded; see README "Progress and findings". Frozen, never retrained against.

# Resolve the project root so this notebook runs unchanged on the DGX Spark (container or
# host), a Mac, or any other checkout: env var first, then the Spark paths, then walk up
# from the working directory until a folder containing docs/PLAN.md is found.
def resolve_project_dir():
    candidates = [
        Path(p) for p in (
            os.environ.get("PI_NN_DIR"),
            "/workspace/training/prompt-injection-nn",
            "/home/spark/projects/training/prompt-injection-nn",
        ) if p
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "docs" / "PLAN.md").exists() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError(f"project root not found; tried {candidates} and parents of {Path.cwd()}")

def resolve_device():
    # CUDA (DGX Spark) > MPS (Apple Silicon) > CPU. Nothing else in the notebook is device-specific.
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def amp_context(device, use_bf16):
    # BF16 autocast only on CUDA. MPS autocast support varies by torch version; CPU gains nothing here.
    if use_bf16 and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def empty_device_cache(device):
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()

# Output contract shared with the LoRA and the Open WebUI filter: "SAFE" or "INJECTION: <reason>".
REASONS = [
    "Override Attempt",
    "Fake System Injection",
    "Meta-Command Injection",
    "Jailbreak Technique",
    "Delimiter Confusion",
    "Privilege Escalation",
    "Embedded Instruction",
    "Role-Play Bypass",
    "Chain-of-Instructions",
]

PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data" / f"source-clean-{DATASET_VERSION}"
MANIFEST_PATH = DATA_DIR / "manifest.json"
assert DATA_DIR.exists(), f"{DATASET_VERSION} splits missing: {DATA_DIR} (run the datagen notebook first)"
manifest = json.loads(MANIFEST_PATH.read_text())
PHASE1_FINGERPRINT = manifest["config_fingerprint"]
assert manifest["reasons"] == REASONS, "reason taxonomy drifted between datagen and this notebook"
REASONS_WITH_TRAINING_ROWS = list(manifest["reasons_with_training_rows"])

SPLIT_PATHS = {name: DATA_DIR / f"{name}.parquet" for name in ("train", "val", "test", "adversarial_test")}
frames = {name: pd.read_parquet(path) for name, path in SPLIT_PATHS.items()}
for name, frame in frames.items():
    expected = manifest["counts"][name]
    assert len(frame) == expected["rows"], (name, len(frame), expected["rows"])
    assert int(frame["label"].sum()) == expected["injection"]
    assert {"text", "label", "reason", "source", "shape", "augmented"}.issubset(frame.columns)
    assert ((frame["label"] == 0) & (frame["reason"] == "")).sum() + ((frame["label"] == 1) & frame["reason"].isin(REASONS)).sum() == len(frame)
assert not frames["val"]["augmented"].any()
assert not frames["test"]["augmented"].any()
assert frames["adversarial_test"]["augmented"].all()
assert frames["adversarial_test"]["label"].eq(1).all()

REASON_INDEX = {reason: i for i, reason in enumerate(REASONS)}
texts = {name: frame["text"].fillna("").astype(str).tolist() for name, frame in frames.items()}
labels = {name: frame["label"].to_numpy(dtype=np.float32) for name, frame in frames.items()}
reason_targets = {name: np.array([REASON_INDEX[r] if r else -1 for r in frame["reason"]], dtype=np.int64) for name, frame in frames.items()}

device = resolve_device()
print({"project": str(PROJECT_DIR), "data": str(DATA_DIR), "torch": torch.__version__, "device": str(device),
       "cuda_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
print({name: len(values) for name, values in texts.items()})
print("reasons with training rows:", REASONS_WITH_TRAINING_ROWS)


{'project': '/workspace/training/prompt-injection-nn', 'data': '/workspace/training/prompt-injection-nn/data/source-clean-v4', 'torch': '2.10.0a0+b558c986e8.nv25.11', 'device': 'cuda', 'cuda_name': 'NVIDIA GB10'}
{'train': 82800, 'val': 9367, 'test': 9456, 'adversarial_test': 3904}
reasons with training rows: ['Chain-of-Instructions', 'Delimiter Confusion', 'Embedded Instruction', 'Fake System Injection', 'Jailbreak Technique', 'Meta-Command Injection', 'Override Attempt', 'Privilege Escalation', 'Role-Play Bypass']


## 2. Metric Helpers

Shared, byte-identical with Phase 2 and the eval notebook, plus the reason-head metrics.


In [2]:
def binary_metrics(y_true, probabilities, threshold):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)) if len(set(y_true)) > 1 else None,
        "average_precision": float(average_precision_score(y_true, probabilities)) if len(set(y_true)) > 1 else None,
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "specificity": float(tn / (tn + fp)) if tn + fp else None,
        "false_positive_rate": float(fp / (tn + fp)) if tn + fp else None,
    }


def adversarial_metrics(probabilities, threshold):
    # The adversarial set is all-positive: only recall and the probability distribution are defined.
    probabilities = np.asarray(probabilities, dtype=np.float64)
    return {
        "threshold": float(threshold),
        "injection_recall": float(np.mean(probabilities >= threshold)),
        "probability_mean": float(probabilities.mean()),
        "probability_median": float(np.median(probabilities)),
        "probability_min": float(probabilities.min()),
        "probability_max": float(probabilities.max()),
    }


def select_threshold(y_true, probabilities):
    # Max validation F1; ties broken by higher recall, then lower threshold.
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    candidates = []
    for index, threshold in enumerate(thresholds):
        denominator = precision[index] + recall[index]
        score = 0.0 if denominator == 0 else 2 * precision[index] * recall[index] / denominator
        candidates.append((score, recall[index], -threshold, threshold))
    return float(max(candidates)[3])


FPR_BUDGETS = (0.01, 0.005, 0.001)


def recall_at_fpr(y_true, probabilities, budgets=FPR_BUDGETS):
    # For each FPR budget: the lowest threshold whose FPR on these rows is within budget, and the
    # recall there. This is the deployment table: how much injection is caught per dropped-benign budget.
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    negatives = np.sort(probabilities[y_true == 0])
    positives = probabilities[y_true == 1]
    table = {}
    for budget in budgets:
        allowed = int(np.floor(budget * len(negatives)))
        # threshold just above the (allowed)-th highest negative score
        threshold = float(np.nextafter(negatives[-allowed - 1], 1.0)) if allowed < len(negatives) else 0.0
        fpr = float(np.mean(negatives >= threshold)) if len(negatives) else 0.0
        table[f"fpr_{budget:g}"] = {
            "threshold": threshold,
            "recall": float(np.mean(positives >= threshold)) if len(positives) else None,
            "achieved_fpr": fpr,
            "allowed_false_positives": allowed,
        }
    return table


def bootstrap_ci(y_true, probabilities, threshold, n_resamples=1000, seed=42, alpha=0.05):
    # Percentile bootstrap over rows. Reported with every locked-test number so a single point
    # estimate is never presented as the result.
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    rng = np.random.default_rng(seed)
    predictions = (probabilities >= threshold).astype(np.int64)
    samples = {"average_precision": [], "roc_auc": [], "f1": [], "recall": [], "false_positive_rate": []}
    n = len(y_true)
    for _ in range(n_resamples):
        index = rng.integers(0, n, n)
        yt, pr, pd_ = y_true[index], probabilities[index], predictions[index]
        if yt.min() == yt.max():
            continue
        tp = int(((pd_ == 1) & (yt == 1)).sum()); fp = int(((pd_ == 1) & (yt == 0)).sum())
        fn = int(((pd_ == 0) & (yt == 1)).sum()); tn = int(((pd_ == 0) & (yt == 0)).sum())
        samples["average_precision"].append(average_precision_score(yt, pr))
        samples["roc_auc"].append(roc_auc_score(yt, pr))
        samples["f1"].append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
        samples["recall"].append(tp / (tp + fn) if tp + fn else 0.0)
        samples["false_positive_rate"].append(fp / (fp + tn) if fp + tn else 0.0)
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    return {
        key: {"low": float(np.percentile(values, lo)), "high": float(np.percentile(values, hi)), "resamples": len(values)}
        for key, values in samples.items()
    }


def slice_metrics(frame, probabilities, threshold, column):
    # Per-slice metrics. Single-class slices (all SAFE or all INJECTION) report only what is defined.
    rows = {}
    probabilities = np.asarray(probabilities, dtype=np.float64)
    for value, index in frame.groupby(column).indices.items():
        y = frame["label"].to_numpy()[index]
        p = probabilities[index]
        predictions = (p >= threshold).astype(int)
        entry = {"rows": int(len(index)), "injection_rows": int(y.sum()), "safe_rows": int((y == 0).sum())}
        if y.sum():
            entry["recall"] = float(predictions[y == 1].mean())
        if (y == 0).sum():
            entry["false_positive_rate"] = float(predictions[y == 0].mean())
        if 0 < y.sum() < len(y):
            entry["f1"] = float(f1_score(y, predictions, zero_division=0))
            entry["average_precision"] = float(average_precision_score(y, p))
        rows[str(value)] = entry
    return rows


def atomic_json(payload, path):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    os.replace(temporary, path)


def reason_metrics(true_reasons, predicted_reasons, reason_logits, defined_reasons):
    # Computed on true positives only. Reasons absent from training are listed as undefined.
    true_reasons = np.asarray(true_reasons); predicted_reasons = np.asarray(predicted_reasons)
    n = len(true_reasons)
    if n == 0:
        return {"rows": 0}
    per_reason = {}
    f1s = []
    for i, reason in enumerate(REASONS):
        support = int((true_reasons == i).sum())
        predicted = int((predicted_reasons == i).sum())
        if reason not in defined_reasons:
            per_reason[reason] = {"support": support, "predicted": predicted, "precision": None, "recall": None, "f1": None, "status": "undefined: no training rows"}
            continue
        tp = int(((true_reasons == i) & (predicted_reasons == i)).sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else None
        f1 = (2 * precision * recall / (precision + recall)) if recall and (precision + recall) else 0.0
        per_reason[reason] = {"support": support, "predicted": predicted, "precision": precision, "recall": recall, "f1": f1 if support else None, "status": "defined"}
        if support:
            f1s.append(f1)
    top2 = np.argsort(-np.asarray(reason_logits), axis=1)[:, :2]
    confusion = confusion_matrix(true_reasons, predicted_reasons, labels=list(range(len(REASONS)))).tolist()
    return {
        "rows": int(n),
        "accuracy": float((predicted_reasons == true_reasons).mean()),
        "top2_accuracy": float(np.mean([t in row for t, row in zip(true_reasons, top2)])),
        "macro_f1_defined_reasons": float(np.mean(f1s)) if f1s else None,
        "defined_reasons": list(defined_reasons),
        "per_reason": per_reason,
        "confusion_rows_true_cols_pred": confusion,
        "confusion_labels": REASONS,
    }

GATE_SHAPES = ("document", "agentic")   # the product's input; chat-shaped prompts are the LoRA's job


def recall_at_fpr_by_shape(frame, probabilities, budgets=FPR_BUDGETS):
    # Threshold chosen within each shape's own rows: an oracle view of how separable each shape is.
    # "gate_rows" pools the GATE_SHAPES rows, the slice the promotion gate is judged on.
    out = {}
    probabilities = np.asarray(probabilities, dtype=np.float64)
    y_all = frame["label"].to_numpy()
    for shape, index in frame.groupby("shape").indices.items():
        y_s = y_all[index]
        if 0 < y_s.sum() < len(y_s):
            out[str(shape)] = recall_at_fpr(y_s, probabilities[index], budgets)
    mask = frame["shape"].isin(GATE_SHAPES).to_numpy()
    if 0 < y_all[mask].sum() < mask.sum():
        out["gate_rows"] = recall_at_fpr(y_all[mask], probabilities[mask], budgets)
    return out


def operating_points(y_val, p_val, frame_test, p_test, budgets=FPR_BUDGETS, val_mask=None):
    # Deployment view: one threshold per budget chosen on val, then reported on test overall, per shape,
    # and on the gate rows. With val_mask the threshold is chosen on a subset of val only; the retrieval
    # filter never sees chat-shaped text, so its operating point is chosen on val document + agentic rows.
    y_val = np.asarray(y_val); p_val = np.asarray(p_val, dtype=np.float64)
    if val_mask is not None:
        y_val, p_val = y_val[val_mask], p_val[val_mask]
    val_table = recall_at_fpr(y_val, p_val, budgets)
    p_test = np.asarray(p_test, dtype=np.float64)
    y_test = frame_test["label"].to_numpy()
    shapes = frame_test["shape"].to_numpy().astype(str)

    def at(threshold, mask):
        y_m, p_m = y_test[mask], p_test[mask]
        pred = p_m >= threshold
        return {"rows": int(mask.sum()),
                "recall": float(pred[y_m == 1].mean()) if (y_m == 1).any() else None,
                "fpr": float(pred[y_m == 0].mean()) if (y_m == 0).any() else None}

    table = {}
    for budget, row in val_table.items():
        threshold = row["threshold"]
        table[budget] = {
            "threshold": threshold, "val_recall": row["recall"], "val_fpr": row["achieved_fpr"],
            "test": at(threshold, np.ones(len(y_test), dtype=bool)),
            "test_by_shape": {s: at(threshold, shapes == s) for s in sorted(set(shapes))},
            "test_gate_rows": at(threshold, np.isin(shapes, GATE_SHAPES)),
        }
    return table


def save_probabilities(frame, probability_columns, path):
    # Per-row probabilities beside the run so a later notebook can align candidates row for row.
    out = frame[["source_id", "label", "shape", "source"]].copy()
    for name, values in probability_columns.items():
        out[name] = np.asarray(values, dtype=np.float64)
    tmp = Path(str(path) + ".tmp")
    out.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def shape_rafpr_rows(tables_by_name):
    # {(name, shape): {budget: recall}} for a Markdown table.
    return {(name, shape): {b: row["recall"] for b, row in table.items()}
            for name, tables in tables_by_name.items() for shape, table in tables.items()}


def retrieval_mask(frame):
    return frame["shape"].isin(GATE_SHAPES).to_numpy()


def retrieval_thresholds(frame_val, p_val, budgets=FPR_BUDGETS):
    # Thresholds for the retrieval path: chosen on val document + agentic rows only.
    mask = retrieval_mask(frame_val)
    return recall_at_fpr(frame_val["label"].to_numpy()[mask], np.asarray(p_val, dtype=np.float64)[mask], budgets)


def operating_point_rows(points_by_name):
    rows = {}
    for name, table in points_by_name.items():
        for budget, e in table.items():
            row = {"threshold": e["threshold"], "val_recall": e["val_recall"], "test_recall": e["test"]["recall"], "test_fpr": e["test"]["fpr"],
                   "gate_recall": e["test_gate_rows"]["recall"], "gate_fpr": e["test_gate_rows"]["fpr"]}
            for s, v in e["test_by_shape"].items():
                row[f"{s}_recall"] = v["recall"]; row[f"{s}_fpr"] = v["fpr"]
            rows[(name, budget)] = row
    return rows


## 3. Tokenizer, Train-Only Vocabulary, Datasets

Cased, punctuation-preserving tokenizer (section comment). The vocabulary is built from train
only. Every row yields a token tensor, a binary label and a reason index (-1 for SAFE, which the
reason loss ignores).


In [3]:
TEXT_CONFIG = {
    "max_vocab": 50000,
    "min_token_frequency": 2,
    "max_tokens": 512,
    "cased": True,
    "embedding_dim": 128,
    "hidden_dim": 128,
    "cnn_channels": 128,
    "dropout": 0.25,
    "batch_size": 256,
    "learning_rate": 2e-3,
    "weight_decay": 1e-4,
    "max_epochs": 20,
    "patience": 4,
    "reason_loss_weight": 0.5,
}

# Cased, punctuation-preserving tokenizer (PLAN.md section 3). Three token classes: letter runs,
# digit runs, and runs of everything else. "<|im_start|>" becomes "<|", "im", "_", "start", "|>";
# "###" and "---" survive as single tokens; "IGNORE" and "ignore" are different tokens.
TOKEN_PATTERN = re.compile(r"[A-Za-z]+|\d+|[^\sA-Za-z\d]+")
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"


def tokenize(text, max_tokens):
    return TOKEN_PATTERN.findall(text)[:max_tokens]


def encode(text, vocab, max_tokens):
    encoded = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text, max_tokens)]
    return torch.tensor(encoded or [vocab[UNK_TOKEN]], dtype=torch.long)

token_counts = Counter(token for text in texts["train"] for token in tokenize(text, TEXT_CONFIG["max_tokens"]))
vocab_tokens = [token for token, count in token_counts.most_common(TEXT_CONFIG["max_vocab"] - 2) if count >= TEXT_CONFIG["min_token_frequency"]]
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, **{token: index + 2 for index, token in enumerate(vocab_tokens)}}
assert len(vocab) <= TEXT_CONFIG["max_vocab"]
PAD_INDEX = vocab[PAD_TOKEN]


class ChunkDataset(Dataset):
    def __init__(self, text_values, label_values, reason_values):
        self.sequences = [encode(text, vocab, TEXT_CONFIG["max_tokens"]) for text in text_values]
        self.labels = torch.tensor(label_values, dtype=torch.float32)
        self.reasons = torch.tensor(reason_values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.sequences[index], self.labels[index], self.reasons[index]


def collate(batch):
    sequences, label_values, reason_values = zip(*batch)
    return pad_sequence(sequences, batch_first=True, padding_value=PAD_INDEX), torch.stack(label_values), torch.stack(reason_values)


datasets = {name: ChunkDataset(texts[name], labels[name], reason_targets[name]) for name in texts}
lengths = np.array([len(s) for s in datasets["train"].sequences])
print({"vocab_size": len(vocab), "train_examples": len(datasets["train"]),
       "train_tokens_median": int(np.median(lengths)), "train_rows_truncated_at_max_tokens": int((lengths >= TEXT_CONFIG["max_tokens"]).sum())})
print("sample tokens:", tokenize(texts["train"][0], 24))


{'vocab_size': 50000, 'train_examples': 82800, 'train_tokens_median': 100, 'train_rows_truncated_at_max_tokens': 4334}
sample tokens: ['"', 'Could', 'you', 'please', 'help', 'me', 'understand', 'and', 'implement', 'the', 'following', 'JSON', 'request', 'that', 'involves', 'extracting', 'user', 'information', 'in', 'a', 'subtle', 'way', ',', 'without']


## 4. Train

Loss = weighted binary cross-entropy on the injection head + `reason_loss_weight` × cross-entropy on
the reason head over positive rows only (`ignore_index=-1`). Early stopping on validation binary log
loss; the best state is restored. BF16 autocast on CUDA only.


In [4]:
class TextCnnBiGru(nn.Module):
    """1D-CNN over token embeddings, bidirectional GRU, masked max-pooling, two heads:
    a binary injection logit and a nine-way reason logit vector."""

    def __init__(self, vocab_size, pad_index, num_reasons, config):
        super().__init__()
        self.pad_index = pad_index
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=pad_index)
        self.cnn = nn.Conv1d(config["embedding_dim"], config["cnn_channels"], kernel_size=3, padding=1)
        self.gru = nn.GRU(config["cnn_channels"], config["hidden_dim"], batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.shared = nn.Sequential(nn.Linear(config["hidden_dim"] * 2, config["hidden_dim"]), nn.ReLU(), nn.Dropout(config["dropout"]))
        self.injection_head = nn.Linear(config["hidden_dim"], 1)
        self.reason_head = nn.Linear(config["hidden_dim"], num_reasons)

    def forward(self, token_ids):
        # Padding-invariant: the convolution's bleed into pad positions is zeroed and the GRU runs on
        # packed sequences, so a text scores identically alone or in any batch. (An earlier version let
        # the backward GRU read the padding, which made scores depend on batch composition.)
        mask = token_ids.ne(self.pad_index)
        lengths = mask.sum(dim=1).clamp(min=1)
        embedded = self.embedding(token_ids).transpose(1, 2)
        features = torch.relu(self.cnn(embedded)).transpose(1, 2)
        features = features.masked_fill(~mask.unsqueeze(-1), 0.0)
        packed = pack_padded_sequence(features, lengths.cpu(), batch_first=True, enforce_sorted=False)
        sequence, _ = self.gru(packed)
        sequence, _ = pad_packed_sequence(sequence, batch_first=True, total_length=token_ids.size(1))
        masked = sequence.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        pooled = self.dropout(masked.max(dim=1).values)
        hidden = self.shared(pooled)
        return self.injection_head(hidden).squeeze(1), self.reason_head(hidden)


def predict(model, dataset, device, use_bf16):
    loader = DataLoader(dataset, batch_size=TEXT_CONFIG["batch_size"], shuffle=False, collate_fn=collate, pin_memory=False)
    probabilities, reason_logits = [], []
    model.eval()
    with torch.inference_mode():
        for token_ids, _, _ in loader:
            token_ids = token_ids.to(device)
            with amp_context(device, use_bf16):
                injection_logit, reason_logit = model(token_ids)
            probabilities.append(torch.sigmoid(injection_logit.float()).cpu().numpy())
            reason_logits.append(reason_logit.float().cpu().numpy())
    return np.concatenate(probabilities), np.concatenate(reason_logits)


use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
model = TextCnnBiGru(len(vocab), PAD_INDEX, len(REASONS), TEXT_CONFIG).to(device)
PARAMETER_COUNT = sum(p.numel() for p in model.parameters())
optimizer = torch.optim.AdamW(model.parameters(), lr=TEXT_CONFIG["learning_rate"], weight_decay=TEXT_CONFIG["weight_decay"])
positive_weight = float((labels["train"] == 0).sum() / (labels["train"] == 1).sum())
injection_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=device))
reason_criterion = nn.CrossEntropyLoss(ignore_index=-1)
train_loader = DataLoader(datasets["train"], batch_size=TEXT_CONFIG["batch_size"], shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), collate_fn=collate, pin_memory=False)
print({"parameters": PARAMETER_COUNT, "positive_weight": round(positive_weight, 4), "device": str(device), "bf16": use_bf16})

best_loss, best_epoch, stale_epochs, best_state = float("inf"), 0, 0, None
history = []
for epoch in range(1, TEXT_CONFIG["max_epochs"] + 1):
    model.train()
    total_loss = total_injection = total_reason = 0.0
    started = time.time()
    for token_ids, batch_labels, batch_reasons in train_loader:
        token_ids, batch_labels, batch_reasons = token_ids.to(device), batch_labels.to(device), batch_reasons.to(device)
        optimizer.zero_grad(set_to_none=True)
        with amp_context(device, use_bf16):
            injection_logit, reason_logit = model(token_ids)
            injection_loss = injection_criterion(injection_logit.float(), batch_labels)
            if (batch_reasons >= 0).any():
                reason_loss = reason_criterion(reason_logit.float(), batch_reasons)
            else:
                reason_loss = torch.zeros((), device=device)
            loss = injection_loss + TEXT_CONFIG["reason_loss_weight"] * reason_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(batch_labels)
        total_injection += injection_loss.item() * len(batch_labels)
        total_reason += reason_loss.item() * len(batch_labels)
    val_probabilities, val_reason_logits = predict(model, datasets["val"], device, use_bf16)
    validation_loss = float(log_loss(labels["val"], val_probabilities, labels=[0, 1]))
    positives = reason_targets["val"] >= 0
    val_reason_accuracy = float((val_reason_logits[positives].argmax(1) == reason_targets["val"][positives]).mean())
    history.append({"epoch": epoch, "train_loss": total_loss / len(datasets["train"]),
                    "train_injection_loss": total_injection / len(datasets["train"]),
                    "train_reason_loss": total_reason / len(datasets["train"]),
                    "validation_log_loss": validation_loss, "validation_reason_accuracy_on_positives": val_reason_accuracy,
                    "seconds": round(time.time() - started, 1)})
    print(history[-1])
    if validation_loss < best_loss - 1e-5:
        best_loss, best_epoch, stale_epochs = validation_loss, epoch, 0
        best_state = deepcopy({key: value.detach().cpu() for key, value in model.state_dict().items()})
    else:
        stale_epochs += 1
    empty_device_cache(device)
    gc.collect()
    if stale_epochs >= TEXT_CONFIG["patience"]:
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(device)
print({"best_epoch": best_epoch, "best_validation_log_loss": best_loss, "epochs_run": len(history)})


{'parameters': 6681610, 'positive_weight': 1.1061, 'device': 'cuda', 'bf16': True}


{'epoch': 1, 'train_loss': 0.5899186989189922, 'train_injection_loss': 0.37704757411410844, 'train_reason_loss': 0.42574225282899425, 'validation_log_loss': 0.24720290747725335, 'validation_reason_accuracy_on_positives': 0.9140167896209616, 'seconds': 118.0}


{'epoch': 2, 'train_loss': 0.29777786373516213, 'train_injection_loss': 0.21008157908628527, 'train_reason_loss': 0.1753925687276223, 'validation_log_loss': 0.20373498421760655, 'validation_reason_accuracy_on_positives': 0.9364029509030781, 'seconds': 117.2}


{'epoch': 3, 'train_loss': 0.20809141028330522, 'train_injection_loss': 0.15319176551656447, 'train_reason_loss': 0.10979928927145143, 'validation_log_loss': 0.19621531074878873, 'validation_reason_accuracy_on_positives': 0.9358941745103028, 'seconds': 117.6}


{'epoch': 4, 'train_loss': 0.15440458537875743, 'train_injection_loss': 0.11515090696765605, 'train_reason_loss': 0.07850735639532407, 'validation_log_loss': 0.20993176644106143, 'validation_reason_accuracy_on_positives': 0.9425082676163826, 'seconds': 117.8}


{'epoch': 5, 'train_loss': 0.1163653232725922, 'train_injection_loss': 0.08768612600755, 'train_reason_loss': 0.057358394486892623, 'validation_log_loss': 0.2193603340600097, 'validation_reason_accuracy_on_positives': 0.9407275502416688, 'seconds': 117.8}


{'epoch': 6, 'train_loss': 0.09077216855425765, 'train_injection_loss': 0.06917573863541446, 'train_reason_loss': 0.04319285980313296, 'validation_log_loss': 0.2202045584807789, 'validation_reason_accuracy_on_positives': 0.9503943017044009, 'seconds': 118.0}


{'epoch': 7, 'train_loss': 0.06905940047784703, 'train_injection_loss': 0.05154488573435712, 'train_reason_loss': 0.035029029501377096, 'validation_log_loss': 0.2582817715428443, 'validation_reason_accuracy_on_positives': 0.9419994912236073, 'seconds': 117.8}
{'best_epoch': 3, 'best_validation_log_loss': 0.19621531074878873, 'epochs_run': 7}


## 5. Select on Validation, Then Evaluate the Locked Holdouts

Threshold by validation F1. Then, once: test with bootstrap intervals, recall at the FPR budgets,
slices by source and shape, reason metrics on true positives, and the adversarial set by variant
kind. `lora_holdout.parquet` is not touched here; `notebooks/eval/` scores it.


In [5]:
val_probabilities, val_reason_logits = predict(model, datasets["val"], device, use_bf16)
threshold = select_threshold(labels["val"], val_probabilities)
validation_metrics = binary_metrics(labels["val"], val_probabilities, threshold)
validation_recall_at_fpr = recall_at_fpr(labels["val"], val_probabilities)


def evaluate_reasons(split, probabilities, reason_logits):
    true_positive = (labels[split] == 1) & (probabilities >= threshold)
    return reason_metrics(reason_targets[split][true_positive], reason_logits[true_positive].argmax(1),
                          reason_logits[true_positive], REASONS_WITH_TRAINING_ROWS)


test_probabilities, test_reason_logits = predict(model, datasets["test"], device, use_bf16)
adversarial_probabilities, adversarial_reason_logits = predict(model, datasets["adversarial_test"], device, use_bf16)

final_metrics = {
    "validation": validation_metrics,
    "validation_recall_at_fpr": validation_recall_at_fpr,
    "validation_reasons_on_true_positives": evaluate_reasons("val", val_probabilities, val_reason_logits),
    "test": binary_metrics(labels["test"], test_probabilities, threshold),
    "test_bootstrap_95ci": bootstrap_ci(labels["test"], test_probabilities, threshold, seed=SEED),
    "test_recall_at_fpr": recall_at_fpr(labels["test"], test_probabilities),
    "test_slices_by_source": slice_metrics(frames["test"], test_probabilities, threshold, "source"),
    "test_slices_by_shape": slice_metrics(frames["test"], test_probabilities, threshold, "shape"),
    "test_reasons_on_true_positives": evaluate_reasons("test", test_probabilities, test_reason_logits),
    "adversarial_test": adversarial_metrics(adversarial_probabilities, threshold),
    "adversarial_test_by_kind": slice_metrics(frames["adversarial_test"], adversarial_probabilities, threshold, "aug_kind"),
}

final_metrics["test_recall_at_fpr_by_shape"] = recall_at_fpr_by_shape(frames["test"], test_probabilities)
final_metrics["operating_points_from_val"] = operating_points(labels["val"], val_probabilities, frames["test"], test_probabilities)

# ---- Candidates and the promotion gate ----
# Phase 2 saves per-row calibrated probabilities beside its run. Aligned row for row, they let this
# notebook score the baseline and a mean ensemble (TF-IDF + CNN) on exactly the same locked rows.
PHASE2_DIR = PROJECT_DIR / "models" / f"phase2-{DATASET_VERSION}-{PHASE1_FINGERPRINT[:16]}"
candidates = {"cnn_bigru": {"val": val_probabilities.astype(np.float64), "test": test_probabilities.astype(np.float64), "adversarial_test": adversarial_probabilities.astype(np.float64)}}
if (PHASE2_DIR / "probabilities_test.parquet").exists():
    phase2_probs = {}
    for split_name in ("val", "test", "adversarial_test"):
        p2 = pd.read_parquet(PHASE2_DIR / f"probabilities_{split_name}.parquet")
        assert len(p2) == len(frames[split_name]), f"Phase 2 probabilities for {split_name}: {len(p2)} rows vs {len(frames[split_name])}"
        assert (p2["source_id"].to_numpy().astype(str) == frames[split_name]["source_id"].to_numpy().astype(str)).all(), f"Phase 2 probabilities for {split_name} are not row-aligned"
        phase2_probs[split_name] = p2["tfidf_logreg"].to_numpy(dtype=np.float64)
    candidates["tfidf_logreg"] = phase2_probs
    candidates["ensemble_mean"] = {s: 0.5 * (candidates["cnn_bigru"][s] + phase2_probs[s]) for s in phase2_probs}
else:
    print("Phase 2 probabilities not found under", PHASE2_DIR, "; run phase_2_baselines first for the gate comparison")


def candidate_tables(p):
    thr = select_threshold(labels["val"], p["val"])
    return {
        "threshold_f1_val": thr,
        "validation": binary_metrics(labels["val"], p["val"], thr),
        "test": binary_metrics(labels["test"], p["test"], thr),
        "test_bootstrap_95ci": bootstrap_ci(labels["test"], p["test"], thr, seed=SEED),
        "test_recall_at_fpr": recall_at_fpr(labels["test"], p["test"]),
        "test_recall_at_fpr_by_shape": recall_at_fpr_by_shape(frames["test"], p["test"]),
        "operating_points_from_val": operating_points(labels["val"], p["val"], frames["test"], p["test"]),
        "operating_points_from_val_retrieval": operating_points(labels["val"], p["val"], frames["test"], p["test"], val_mask=retrieval_mask(frames["val"])),
        "fpr_budget_thresholds_val_retrieval": retrieval_thresholds(frames["val"], p["val"]),
        "test_slices_by_shape": slice_metrics(frames["test"], p["test"], thr, "shape"),
        "test_slices_by_source": slice_metrics(frames["test"], p["test"], thr, "source"),
        "adversarial_test": adversarial_metrics(p["adversarial_test"], thr),
        "fpr_budget_thresholds_val": recall_at_fpr(labels["val"], p["val"]),
    }


candidate_metrics = {name: candidate_tables(p) for name, p in candidates.items()}
final_metrics["candidates"] = candidate_metrics

# The gate (PLAN.md section 4), judged on the product's rows (document + agentic). A candidate clears it
# when its recall at every FPR budget on those rows (threshold chosen within those rows) is at least the
# baseline's, and its adversarial recall is within one point of the baseline's. The recommended backend
# is the clearing candidate with the highest gate-row recall at the 1% budget, else the baseline.
gate = {"shapes": list(GATE_SHAPES), "budgets": [f"fpr_{b:g}" for b in FPR_BUDGETS], "baseline": "tfidf_logreg", "verdicts": {}}
if "tfidf_logreg" in candidate_metrics:
    base = candidate_metrics["tfidf_logreg"]
    for name, cm in candidate_metrics.items():
        if name == "tfidf_logreg":
            continue
        budgets = {}
        for b in gate["budgets"]:
            cand_r = cm["test_recall_at_fpr_by_shape"]["gate_rows"][b]["recall"]
            base_r = base["test_recall_at_fpr_by_shape"]["gate_rows"][b]["recall"]
            budgets[b] = {"candidate": cand_r, "baseline": base_r, "clears": bool(cand_r >= base_r)}
        adversarial_ok = bool(cm["adversarial_test"]["injection_recall"] >= base["adversarial_test"]["injection_recall"] - 0.01)
        gate["verdicts"][name] = {"budgets": budgets, "adversarial_ok": adversarial_ok,
                                  "clears_gate": bool(all(r["clears"] for r in budgets.values()) and adversarial_ok)}
    clearing = [n for n, v in gate["verdicts"].items() if v["clears_gate"]]
    gate["recommended_backend"] = (max(clearing, key=lambda n: candidate_metrics[n]["test_recall_at_fpr_by_shape"]["gate_rows"]["fpr_0.01"]["recall"])
                                   if clearing else "tfidf_logreg")
else:
    gate["recommended_backend"] = "cnn_bigru"
    gate["note"] = "Phase 2 probabilities absent; gate not evaluated"
final_metrics["gate"] = gate

print("CANDIDATES on gate rows (document + agentic), recall at FPR budget, threshold chosen within those rows:")
print(pd.DataFrame({name: {b: cm["test_recall_at_fpr_by_shape"]["gate_rows"][b]["recall"] for b in gate["budgets"]} for name, cm in candidate_metrics.items()}).T)
print("adversarial recall:", {name: round(cm["adversarial_test"]["injection_recall"], 4) for name, cm in candidate_metrics.items()})
print("GATE verdicts:", {name: v["clears_gate"] for name, v in gate["verdicts"].items()}, "| recommended backend:", gate["recommended_backend"])
print("retrieval operating points (thresholds from val document+agentic rows; test gate rows):")
print(pd.DataFrame({(n, b): {"threshold": e["threshold"], "gate_recall": e["test_gate_rows"]["recall"], "gate_fpr": e["test_gate_rows"]["fpr"]} for n, cm in candidate_metrics.items() for b, e in cm["operating_points_from_val_retrieval"].items()}).T)
print()
print("VALIDATION", {k: round(v, 4) for k, v in validation_metrics.items() if isinstance(v, float)})
print()
print("LOCKED TEST", {k: round(v, 4) for k, v in final_metrics["test"].items() if isinstance(v, float)})
print("95% CI:", {k: (round(v["low"], 4), round(v["high"], 4)) for k, v in final_metrics["test_bootstrap_95ci"].items()})
print("recall at FPR budget:", {k: round(v["recall"], 4) for k, v in final_metrics["test_recall_at_fpr"].items()})
print()
print("test by shape:", {k: {m: round(v, 4) for m, v in row.items() if m in ("recall", "false_positive_rate", "f1")} for k, row in final_metrics["test_slices_by_shape"].items()})
print("test by source:")
for source, row in final_metrics["test_slices_by_source"].items():
    print(f"  {source:32s} rows={row['rows']:>6,}  " + "  ".join(f"{m}={row[m]:.4f}" for m in ("recall", "false_positive_rate", "f1") if m in row))
print()
reasons_report = final_metrics["test_reasons_on_true_positives"]
print("reason head on test true positives:", {k: reasons_report[k] for k in ("rows", "accuracy", "top2_accuracy", "macro_f1_defined_reasons")})
for reason, row in reasons_report["per_reason"].items():
    print(f"  {reason:24s} support={row['support']:>5}  predicted={row['predicted']:>5}  " + (f"P={row['precision']:.3f} R={row['recall']:.3f}" if row["status"] == "defined" and row["recall"] is not None else row["status"]))
print()
print("LOCKED ADVERSARIAL TEST", {k: round(v, 4) for k, v in final_metrics["adversarial_test"].items() if isinstance(v, float)})
print("by kind:", {k: round(row["recall"], 4) for k, row in final_metrics["adversarial_test_by_kind"].items()})


CANDIDATES on gate rows (document + agentic), recall at FPR budget, threshold chosen within those rows:
               fpr_0.01  fpr_0.005  fpr_0.001
cnn_bigru      0.954232   0.929781   0.889655
tfidf_logreg   0.907837   0.867085   0.843260
ensemble_mean  0.960502   0.942947   0.908464
adversarial recall: {'cnn_bigru': 0.9649, 'tfidf_logreg': 0.9664, 'ensemble_mean': 0.9672}
GATE verdicts: {'cnn_bigru': True, 'ensemble_mean': True} | recommended backend: ensemble_mean
retrieval operating points (thresholds from val document+agentic rows; test gate rows):
                         threshold  gate_recall  gate_fpr
cnn_bigru     fpr_0.01    0.406862     0.955486  0.012250
              fpr_0.005   0.564579     0.934796  0.005803
              fpr_0.001   0.856812     0.897179  0.001289
tfidf_logreg  fpr_0.01    0.657664     0.926019  0.014184
              fpr_0.005   0.806223     0.880878  0.005158
              fpr_0.001   0.892830     0.840125  0.000645
ensemble_mean fpr_0.01    0.4546

## 6. Efficiency: Parameters, Size, CPU Latency

The filter runs this model on the CPU of whatever hosts Open WebUI, on the critical path of every
turn with retrieval. Measured here on a copy of the model on CPU: batch of ten 512-token chunks and
batch of one, p50 and p95 over 1,000 warm calls (`docs/EVALUATION.md` §8, target p95 under 100 ms
for the batch of ten).


In [6]:
cpu_model = deepcopy(model).to("cpu").eval()
torch.set_num_threads(max(1, os.cpu_count() // 2))


def time_batch(batch_size, tokens, calls=1000, warmup=50):
    token_ids = torch.randint(2, len(vocab), (batch_size, tokens), dtype=torch.long)
    with torch.inference_mode():
        for _ in range(warmup):
            cpu_model(token_ids)
        timings = []
        for _ in range(calls):
            started = time.perf_counter()
            cpu_model(token_ids)
            timings.append((time.perf_counter() - started) * 1000)
    return {"batch_size": batch_size, "tokens": tokens, "calls": calls,
            "p50_ms": float(np.percentile(timings, 50)), "p95_ms": float(np.percentile(timings, 95))}


latency = {
    "batch_10_x_512": time_batch(10, TEXT_CONFIG["max_tokens"]),
    "batch_1_x_512": time_batch(1, TEXT_CONFIG["max_tokens"]),
    "batch_32_x_512": time_batch(32, TEXT_CONFIG["max_tokens"], calls=200),
    "cpu": platform.processor() or platform.machine(),
    "torch_threads": torch.get_num_threads(),
}
latency["batch_32_x_512"]["chunks_per_second"] = 32 * 1000 / latency["batch_32_x_512"]["p50_ms"]
print({"parameters": PARAMETER_COUNT})
for key in ("batch_10_x_512", "batch_1_x_512", "batch_32_x_512"):
    print(key, {k: (round(v, 2) if isinstance(v, float) else v) for k, v in latency[key].items()})
print("p95 target for batch of ten met:", latency["batch_10_x_512"]["p95_ms"] < 100)


{'parameters': 6681610}
batch_10_x_512 {'batch_size': 10, 'tokens': 512, 'calls': 1000, 'p50_ms': 28.45, 'p95_ms': 36.59}
batch_1_x_512 {'batch_size': 1, 'tokens': 512, 'calls': 1000, 'p50_ms': 12.85, 'p95_ms': 15.15}
batch_32_x_512 {'batch_size': 32, 'tokens': 512, 'calls': 200, 'p50_ms': 111.37, 'p95_ms': 122.61, 'chunks_per_second': 287.33}
p95 target for batch of ten met: True


## 7. Save Phase 3 Artifacts

Weights, vocabulary, tokenizer config, thresholds, metrics, history, config, package versions and the run manifest, keyed by the dataset fingerprint. The directory must not already exist: a rerun is a new run.


In [7]:
run_id = f"phase3-cnn-bigru-{DATASET_VERSION}-{PHASE1_FINGERPRINT[:16]}"
output_dir = PROJECT_DIR / "models" / run_id
output_dir.mkdir(parents=True, exist_ok=False)

torch.save(best_state, output_dir / "model_state.pt")
(output_dir / "vocab.json").write_text(json.dumps(vocab, sort_keys=True, ensure_ascii=False))
atomic_json({"token_pattern": TOKEN_PATTERN.pattern, "cased": TEXT_CONFIG["cased"], "max_tokens": TEXT_CONFIG["max_tokens"],
             "pad_token": PAD_TOKEN, "unk_token": UNK_TOKEN, "pad_index": PAD_INDEX, "unk_index": vocab[UNK_TOKEN],
             "vocab_size": len(vocab)}, output_dir / "tokenizer.json")
atomic_json({"threshold": threshold, "selected_on": "val.parquet, max F1",
             "fpr_budget_thresholds_val": validation_recall_at_fpr,
             "candidates": {name: {"threshold_f1_val": cm["threshold_f1_val"], "fpr_budget_thresholds_val": cm["fpr_budget_thresholds_val"],
                                   "fpr_budget_thresholds_val_retrieval": cm["fpr_budget_thresholds_val_retrieval"]} for name, cm in candidate_metrics.items()},
             "fpr_budget_thresholds_val_retrieval": candidate_metrics["cnn_bigru"]["fpr_budget_thresholds_val_retrieval"],
             "note_retrieval": "fpr_budget_thresholds_val_retrieval: chosen on val document + agentic rows, the retrieval filter's input; the filter ships with these",
             "recommended_backend": gate["recommended_backend"],
             "note": "for deployment choose the threshold from the recall-at-FPR table against the operator's FPR budget"}, output_dir / "thresholds.json")
atomic_json(final_metrics, output_dir / "metrics.json")
atomic_json(gate, output_dir / "gate.json")
for split_name, probs in (("val", val_probabilities), ("test", test_probabilities), ("adversarial_test", adversarial_probabilities)):
    save_probabilities(frames[split_name], {name: p[split_name] for name, p in candidates.items()}, output_dir / f"probabilities_{split_name}.parquet")
atomic_json(history, output_dir / "training_history.json")
atomic_json({"seed": SEED, "text": TEXT_CONFIG, "reasons": REASONS, "architecture": "CNN(k=3) + packed BiGRU + masked max-pool, injection head + reason head (padding-invariant)",
             "parameters": PARAMETER_COUNT, "best_epoch": best_epoch, "positive_weight": positive_weight}, output_dir / "config.json")
atomic_json({
    "python": platform.python_version(), "numpy": package_version("numpy"), "pandas": package_version("pandas"),
    "scikit_learn": package_version("scikit-learn"), "torch": package_version("torch"),
    "cuda_available": torch.cuda.is_available(), "cuda_version": torch.version.cuda,
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "mps_available": bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()),
    "device": str(device), "bf16": use_bf16,
}, output_dir / "package_versions.json")
model_size_bytes = (output_dir / "model_state.pt").stat().st_size
atomic_json({
    "run_id": run_id, "dataset_version": DATASET_VERSION, "created_utc": datetime.now(timezone.utc).isoformat(),
    "phase1_config_fingerprint": PHASE1_FINGERPRINT, "phase1_manifest": str(MANIFEST_PATH),
    "source_files": {name: str(path) for name, path in SPLIT_PATHS.items()},
    "source_counts": {name: {k: v for k, v in counts.items() if not isinstance(v, dict)} for name, counts in manifest["counts"].items()},
    "reasons_with_training_rows": REASONS_WITH_TRAINING_ROWS,
    "chosen_model": "cnn_bigru_two_head", "selection_data": "val.parquet only",
    "selection_criteria": ["early stopping on validation binary log loss", "threshold by validation F1"],
    "holdout_policy": "test and adversarial_test opened once after selection; lora_holdout scored only by notebooks/eval",
    "parameters": PARAMETER_COUNT, "model_state_bytes": model_size_bytes, "latency": latency,
}, output_dir / "run_manifest.json")
print(output_dir)
print({"parameters": PARAMETER_COUNT, "model_state_mb": round(model_size_bytes / 1e6, 2), "threshold": round(threshold, 4),
       "test_f1": round(final_metrics["test"]["f1"], 4), "test_fpr": round(final_metrics["test"]["false_positive_rate"], 4),
       "adversarial_recall": round(final_metrics["adversarial_test"]["injection_recall"], 4)})


/workspace/training/prompt-injection-nn/models/phase3-cnn-bigru-v4-7f42d9d83806880d
{'parameters': 6681610, 'model_state_mb': 26.73, 'threshold': 0.3242, 'test_f1': 0.8961, 'test_fpr': 0.0994, 'adversarial_recall': 0.9649}


## 8. Report

Writes `reports/<UTC stamp>_phase_3_cnn_bigru_<version>_<fingerprint>/report.md` with the model summary, training
curve, validation and locked-test tables with bootstrap intervals, recall at the FPR budgets,
slices, the reason-head table, the adversarial set by variant, and the CPU latency table. The
training curve is rendered here and saved beside it.


In [8]:
# ---- Report: tracked Markdown summary of this run, written to reports/ with its plots beside it ----
import shutil

REPORTS_DIR = PROJECT_DIR / "reports"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%SZ")


def report_dir(run_name):
    # One folder per run: reports/<UTC stamp>_<phase>_<version>_<fingerprint>/ holding report.md and its plots.
    path = REPORTS_DIR / f"{RUN_STAMP}_{run_name}"
    path.mkdir(parents=True, exist_ok=True)
    return path


def md_table(frame, floatfmt="{:.4f}", index=True):
    frame = pd.DataFrame(frame)
    cols = [str(c) for c in frame.columns]
    header = ([""] if index else []) + cols
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for idx, row in frame.iterrows():
        cells = []
        for v in row.tolist():
            if v is None or (isinstance(v, float) and np.isnan(v)):
                cells.append("")
            elif isinstance(v, (float, np.floating)):
                cells.append(floatfmt.format(float(v)))
            elif isinstance(v, (bool, np.bool_)):
                cells.append("yes" if v else "no")
            else:
                cells.append(str(v).replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(([str(idx)] if index else []) + cells) + " |")
    return "\n".join(lines)


def write_report(name, title, sections):
    # sections: list of (heading, markdown body). Atomic write, LF endings.
    parts = [f"# {title}", f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_"]
    for heading, body in sections:
        parts.append(f"## {heading}\n\n{body}" if heading else body)
    path = report_dir(name) / "report.md"
    tmp = Path(str(path) + ".tmp")
    tmp.write_text("\n\n".join(parts) + "\n", encoding="utf-8", newline="\n")
    os.replace(tmp, path)
    print("report written:", path)
    return path


def report_asset(run_name, filename):
    # Plots live beside report.md and are referenced by bare file name.
    return report_dir(run_name), filename

run_name = f"phase_3_cnn_bigru_{DATASET_VERSION}_{PHASE1_FINGERPRINT[:16]}"
curve_line = "_matplotlib not installed in this kernel; chart skipped._"
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    asset_dir, rel = report_asset(run_name, "training_curve.png")
    hist = pd.DataFrame(history)
    figure, axis = plt.subplots(figsize=(8, 4.5))
    axis.plot(hist["epoch"], hist["train_injection_loss"], marker="o", label="train injection loss")
    axis.plot(hist["epoch"], hist["validation_log_loss"], marker="o", label="validation log loss")
    axis.axvline(best_epoch, linestyle="--", color="gray", label=f"best epoch {best_epoch}")
    axis.set(xlabel="epoch", ylabel="loss", title="Training curve")
    axis.legend()
    figure.tight_layout()
    figure.savefig(asset_dir / "training_curve.png", dpi=140)
    plt.close(figure)
    curve_line = f"![training curve]({rel})"
except Exception as exc:
    curve_line = f"_chart skipped: {type(exc).__name__}: {exc}_"
    print("chart skipped:", exc)

headline_cols = ["threshold", "precision", "recall", "f1", "roc_auc", "average_precision", "brier", "log_loss", "false_positive_rate"]
headline = pd.DataFrame({"validation": final_metrics["validation"], "test": final_metrics["test"]}).T[headline_cols]
ci = {k: f"{v['low']:.4f} to {v['high']:.4f}" for k, v in final_metrics["test_bootstrap_95ci"].items()}
rafpr = {budget: {"recall": row["recall"], "threshold": row["threshold"], "achieved_fpr": row["achieved_fpr"]} for budget, row in final_metrics["test_recall_at_fpr"].items()}
shape_rows = {shape: {m: row.get(m) for m in ("rows", "recall", "false_positive_rate", "f1")} for shape, row in final_metrics["test_slices_by_shape"].items()}
source_rows = {source: {m: row.get(m) for m in ("rows", "recall", "false_positive_rate", "f1")} for source, row in final_metrics["test_slices_by_source"].items()}
reasons_report = final_metrics["test_reasons_on_true_positives"]
reason_rows = {reason: {"support": row["support"], "predicted": row["predicted"], "precision": row["precision"], "recall": row["recall"], "status": row["status"]}
               for reason, row in reasons_report["per_reason"].items()}
adv_kind = {kind: {"rows": row["rows"], "recall": row.get("recall")} for kind, row in final_metrics["adversarial_test_by_kind"].items()}
latency_rows = {key: {k: v for k, v in latency[key].items() if k != "calls"} for key in ("batch_10_x_512", "batch_1_x_512", "batch_32_x_512")}

cand_cols = ["threshold", "precision", "recall", "f1", "roc_auc", "average_precision", "false_positive_rate"]
cand_headline = pd.DataFrame({name: cm["test"] for name, cm in candidate_metrics.items()}).T[cand_cols]
cand_rafpr = pd.DataFrame({name: {b: row["recall"] for b, row in cm["test_recall_at_fpr"].items()} for name, cm in candidate_metrics.items()}).T
cand_shape_rafpr = shape_rafpr_rows({name: cm["test_recall_at_fpr_by_shape"] for name, cm in candidate_metrics.items()})
cand_ops = operating_point_rows({name: cm["operating_points_from_val"] for name, cm in candidate_metrics.items()})
cand_ops_retrieval = operating_point_rows({name: cm["operating_points_from_val_retrieval"] for name, cm in candidate_metrics.items()})
cand_shape_slices = {(name, shape): {m: row.get(m) for m in ("rows", "recall", "false_positive_rate")} for name, cm in candidate_metrics.items() for shape, row in cm["test_slices_by_shape"].items()}
cand_adv = {name: {"injection_recall": cm["adversarial_test"]["injection_recall"]} for name, cm in candidate_metrics.items()}
gate_lines = [f"- Judged on test rows of shape {', '.join(gate['shapes'])} (`gate_rows`), baseline `{gate['baseline']}`, budgets {', '.join(gate['budgets'])}.",
              "- A candidate clears the gate when its gate-row recall at every budget is at least the baseline's and its adversarial recall is within one point of the baseline's."]
for name, v in gate.get("verdicts", {}).items():
    gate_lines.append(f"- **{name}**: " + ", ".join(f"{b} {r['candidate']:.3f} vs {r['baseline']:.3f} ({'ok' if r['clears'] else 'below'})" for b, r in v["budgets"].items()) + f"; adversarial {'ok' if v['adversarial_ok'] else 'below'}; **{'CLEARS' if v['clears_gate'] else 'does not clear'}**")
gate_lines.append(f"- **Recommended backend: `{gate['recommended_backend']}`**" + (f" ({gate['note']})" if gate.get("note") else ""))
write_report(run_name, f"Phase 3 Report: CNN + BiGRU on dataset {DATASET_VERSION} ({PHASE1_FINGERPRINT[:16]})", [
    ("Run", f"- Run ID: `{run_id}`\n- Dataset fingerprint: `{PHASE1_FINGERPRINT}`\n- Parameters: {PARAMETER_COUNT:,}; weights on disk: {model_size_bytes / 1e6:.1f} MB\n- Config: vocab {len(vocab):,}, {TEXT_CONFIG['max_tokens']} tokens, embedding {TEXT_CONFIG['embedding_dim']}, hidden {TEXT_CONFIG['hidden_dim']}, cnn {TEXT_CONFIG['cnn_channels']}, dropout {TEXT_CONFIG['dropout']}, batch {TEXT_CONFIG['batch_size']}, lr {TEXT_CONFIG['learning_rate']}, reason loss weight {TEXT_CONFIG['reason_loss_weight']}\n- Best epoch {best_epoch} of {len(history)} run (patience {TEXT_CONFIG['patience']}); threshold {threshold:.4f} by validation F1\n- Device: {device}, bf16 {use_bf16}, torch {torch.__version__}\n- Artifacts: `{output_dir.relative_to(PROJECT_DIR)}`"),
    ("Training curve", md_table(pd.DataFrame(history).set_index("epoch")) + "\n\n" + curve_line),
    ("Headline metrics", md_table(headline) + "\n\n95% bootstrap intervals on test (1,000 resamples):\n\n" + md_table(pd.DataFrame({"test": ci}).T)),
    ("Recall at FPR budget (test)", md_table(pd.DataFrame(rafpr).T)),
    ("Test slices by shape", md_table(pd.DataFrame(shape_rows).T)),
    ("Gate verdict", "\n".join(gate_lines)),
    ("Candidates on the locked test (F1-max threshold from val)", md_table(cand_headline) + "\n\nRecall at FPR budget, all test rows:\n\n" + md_table(cand_rafpr) + "\n\nAdversarial recall:\n\n" + md_table(pd.DataFrame(cand_adv).T)),
    ("Candidates: recall at FPR budget by shape (threshold chosen within the shape)", "`gate_rows` pools the document and agentic rows.\n\n" + md_table(pd.DataFrame(cand_shape_rafpr).T)),
    ("Candidates: operating points chosen on val, reported on test", "One threshold per budget chosen on validation over all rows, applied to the locked test: overall, on the gate rows, and per shape.\n\n" + md_table(pd.DataFrame(cand_ops).T)),
    ("Candidates: retrieval operating points (thresholds chosen on val document + agentic rows)", "The retrieval filter never sees chat-shaped text, so its thresholds are chosen on validation document and agentic rows only. `gate_recall` / `gate_fpr` on the locked test are the numbers the filter ships with.\n\n" + md_table(pd.DataFrame(cand_ops_retrieval).T)),
    ("Candidates: test slices by shape (F1-max threshold)", md_table(pd.DataFrame(cand_shape_slices).T)),
    ("Test slices by source", md_table(pd.DataFrame(source_rows).T)),
    ("Reason head on test true positives", f"- Rows: {reasons_report.get('rows', 0)}; accuracy {reasons_report.get('accuracy')}; top-2 accuracy {reasons_report.get('top2_accuracy')}; macro-F1 over defined reasons {reasons_report.get('macro_f1_defined_reasons')}\n- Reasons with training rows: {', '.join(REASONS_WITH_TRAINING_ROWS)}\n\n" + md_table(pd.DataFrame(reason_rows).T)),
    ("Locked adversarial test", md_table(pd.DataFrame({"adversarial_test": final_metrics["adversarial_test"]}).T) + "\n\nBy variant kind:\n\n" + md_table(pd.DataFrame(adv_kind).T)),
    ("CPU latency", f"CPU: {latency['cpu']}, {latency['torch_threads']} torch threads. Target: p95 under 100 ms for the batch of ten; met: {'yes' if latency['batch_10_x_512']['p95_ms'] < 100 else 'no'}.\n\n" + md_table(pd.DataFrame(latency_rows).T, floatfmt="{:.2f}")),
])


report written: /workspace/training/prompt-injection-nn/reports/2026-10-10_125545Z_phase_3_cnn_bigru_v4_7f42d9d83806880d/report.md


PosixPath('/workspace/training/prompt-injection-nn/reports/2026-10-10_125545Z_phase_3_cnn_bigru_v4_7f42d9d83806880d/report.md')